# 05 - CitrusNet architecture

Defining the model and running sanity checks before the training loop notebook. No pretrained weights anywhere, everything below is trained from scratch.

In [1]:
import os
import time
import importlib
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

## Model code

Writing citrus_model.py to disk so it can be imported here and reused by the training notebook later.

In [2]:
%%writefile citrus_model.py
"""
CitrusNet architecture.

Everything is trained from scratch - no pretrained weights, no backbones,
no transfer learning.
"""

import torch.nn as nn


class CitrusNet(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        # Block 1 - plain conv stack
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        # Block 2 - depthwise separable conv
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=3, padding=1, groups=32),  # depthwise
            nn.Conv2d(32, 64, kernel_size=1),  # pointwise
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        # Block 3 - dilated conv, padding=2 keeps spatial size unchanged
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=2, dilation=2),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        # Block 4 - no pooling, feeds straight into GAP
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.flatten = nn.Flatten()

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x)
        x = self.flatten(x)
        x = self.classifier(x)
        return x


def init_weights(model):
    """
    Conv2d: He/Kaiming normal (relu nonlinearity), bias 0.
    BatchNorm2d: weight 1, bias 0.
    Linear(256, 128): He/Kaiming normal (followed by ReLU).
    Linear(128, num_classes): Xavier/Glorot uniform (feeds straight into the loss).
    """
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            if m.bias is not None:
                nn.init.zeros_(m.bias)
            nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.zeros_(m.bias)

    # classifier[0] = Linear(256, 128), followed by ReLU
    # classifier[-1] = Linear(128, num_classes), feeds into CrossEntropyLoss
    nn.init.kaiming_normal_(model.classifier[0].weight, nonlinearity="relu")
    nn.init.xavier_uniform_(model.classifier[-1].weight)

Overwriting citrus_model.py


In [3]:
import citrus_model
importlib.reload(citrus_model)
from citrus_model import CitrusNet, init_weights

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [5]:
model = CitrusNet(num_classes=4)
init_weights(model)
model = model.to(device)

## Model structure

In [6]:
print(model)

CitrusNet(
  (block1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32)
    (1): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1))
    (2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), pad

## Parameter counts

Total, trainable, and a per-block breakdown from named_parameters().

In [7]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

block_params = defaultdict(int)
for name, p in model.named_parameters():
    block_params[name.split(".")[0]] += p.numel()

label_map = {
    "block1": "Block 1",
    "block2": "Block 2 (depthwise separable)",
    "block3": "Block 3 (dilated)",
    "block4": "Block 4",
    "classifier": "Classifier head",
}

print()
for key in ["block1", "block2", "block3", "block4", "classifier"]:
    print(f"{label_map[key]}: {block_params[key]:,} parameters")

Total parameters: 416,036
Trainable parameters: 416,036

Block 1: 10,272 parameters
Block 2 (depthwise separable): 2,560 parameters
Block 3 (dilated): 74,112 parameters
Block 4: 295,680 parameters
Classifier head: 33,412 parameters


## Weight init check

Spot-checking one conv layer and the output linear layer. Neither should be all zero or NaN.

In [8]:
conv_layer = model.block1[0]
final_linear = model.classifier[-1]

conv_w = conv_layer.weight.detach().cpu()
lin_w = final_linear.weight.detach().cpu()

print(f"Block1 first conv weight - mean: {conv_w.mean().item():.6f}, std: {conv_w.std().item():.6f}")
print(f"Final linear weight - mean: {lin_w.mean().item():.6f}, std: {lin_w.std().item():.6f}")

assert not torch.isnan(conv_w).any(), "conv weights contain NaN"
assert not torch.isnan(lin_w).any(), "linear weights contain NaN"
assert conv_w.abs().sum().item() != 0, "conv weights are all zero"
assert lin_w.abs().sum().item() != 0, "linear weights are all zero"

print("Weight init sanity check passed.")

Block1 first conv weight - mean: 0.006094, std: 0.271026
Final linear weight - mean: 0.001603, std: 0.124626
Weight init sanity check passed.


## Dummy forward pass

GAP collapses every feature map to 1x1 before the classifier, so the same model works on any input resolution without changes. Checked here with 224x224 and 128x128 random tensors.

In [9]:
model.eval()

with torch.no_grad():
    dummy_224 = torch.randn(4, 3, 224, 224, device=device)
    out_224 = model(dummy_224)
    print("Input 224x224 -> output shape:", tuple(out_224.shape))
    assert out_224.shape == (4, 4)

    dummy_128 = torch.randn(4, 3, 128, 128, device=device)
    out_128 = model(dummy_128)
    print("Input 128x128 -> output shape:", tuple(out_128.shape))
    assert out_128.shape == (4, 4)

print("Both dummy input checks passed.")

Input 224x224 -> output shape: (4, 4)
Input 128x128 -> output shape: (4, 4)
Both dummy input checks passed.


## Forward pass on real data

Same check but with an actual batch from the training set, to confirm this isn't just working on random noise.

In [10]:
PROJECT_ROOT = r"C:\workstation\3RD YEAR\citrus_fruit"
DATA_ROOT_224 = os.path.join(PROJECT_ROOT, "processed", "224")

from citrus_common import load_normalization_stats, get_transforms, CitrusLeafDataset

mean, std = load_normalization_stats()
train_transform = get_transforms(augment=True, mean=mean, std=std, size=224)

train_dataset = CitrusLeafDataset(root_dir=DATA_ROOT_224, split="train", transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

images, labels = next(iter(train_loader))
images = images.to(device)

model.eval()
with torch.no_grad():
    logits = model(images)

print("Real batch output shape:", tuple(logits.shape))
print("Logits, first 2 samples:")
print(logits[:2])

Real batch output shape: (8, 4)
Logits, first 2 samples:
tensor([[ 3.1235,  1.7618,  2.4620, -4.2230],
        [ 3.3460,  2.1559,  2.5398, -5.0362]], device='cuda:0')


## CPU vs GPU timing

20 forward passes of a batch of 32 at 224x224, on CPU and on GPU.

In [11]:
if torch.cuda.is_available():
    n_runs = 20
    batch = torch.randn(32, 3, 224, 224)

    model_cpu = model.to("cpu").eval()
    batch_cpu = batch.to("cpu")
    with torch.no_grad():
        start = time.time()
        for _ in range(n_runs):
            _ = model_cpu(batch_cpu)
        cpu_time = time.time() - start

    model_gpu = model.to("cuda").eval()
    batch_gpu = batch.to("cuda")
    with torch.no_grad():
        torch.cuda.synchronize()
        start = time.time()
        for _ in range(n_runs):
            _ = model_gpu(batch_gpu)
        torch.cuda.synchronize()
        gpu_time = time.time() - start

    print(f"CPU time for {n_runs} forward passes: {cpu_time:.4f} s")
    print(f"GPU time for {n_runs} forward passes: {gpu_time:.4f} s")
    print(f"Speedup: {cpu_time / gpu_time:.2f}x")

    model = model.to(device)
else:
    print("CUDA not available, skipping CPU vs GPU timing.")

CPU time for 20 forward passes: 20.3278 s
GPU time for 20 forward passes: 1.7174 s
Speedup: 11.84x


## Summary

In [12]:
model_size_mb = total_params * 4 / 1024**2

print("=" * 40)
print("CitrusNet architecture summary")
print("=" * 40)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size (float32): {model_size_mb:.2f} MB")
print("Dummy input checks (224x224, 128x128): passed")
print("Real batch forward pass: passed")

CitrusNet architecture summary
Total parameters: 416,036
Trainable parameters: 416,036
Model size (float32): 1.59 MB
Dummy input checks (224x224, 128x128): passed
Real batch forward pass: passed
